# Multimodal 3D Generation - Google Colab Training

Notebook này giúp:
1. Mount Google Drive và thiết lập đường dẫn
2. Kiểm tra project code đầy đủ
3. Tải dữ liệu Cap3D ABO (~40GB)
4. Tự động cấu hình training theo GPU (A100/V100/T4/CPU)
5. Train model với auto-save checkpoint vào Drive
6. Evaluate metrics (Chamfer Distance, F-Score)
7. Demo visualization: Point Cloud + Mesh reconstruction
8. Gradio demo với UI chuyên nghiệp

**Khuyến nghị**: Runtime → Change runtime type → **A100 GPU** + High-RAM

## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive mounted')
except:
    print('✓ Google Drive already mounted')

# Kiểm tra GPU
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('✓ GPU available')
except:
    print('⚠ KHÔNG CÓ GPU! Vui lòng chọn Runtime → Change runtime type → T4 GPU')

In [ ]:
import os

# === CẤU HÌNH ĐƯỜNG DẪN ===
# Mọi thứ chạy trực tiếp trên Drive
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'

# Tạo thư mục nếu chưa có
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

print(f'✓ Work dir: {WORK_DIR}')

## 2. Kiểm tra Project Code

Project code nằm trực tiếp trên Drive.  
Đảm bảo đã upload các thư mục sau lên Drive: `src/`, `configs/`, `images/`

In [ ]:
# Kiểm tra xem có đủ files không
import os

required_files = {
    'src/__init__.py': 'Package init',
    'src/train.py': 'Training script',
    'src/evaluate.py': 'Evaluation script',
    'src/demo.py': 'Gradio demo',
    'src/data/__init__.py': 'Data package init',
    'src/data/dataset.py': 'Dataset & DataLoader',
    'src/data/prepare_data.py': 'Data preparation',
    'src/data/transforms.py': 'Data transforms',
    'src/models/__init__.py': 'Models package init',
    'src/models/multimodal3d.py': 'Main model',
    'src/models/encoders.py': 'CLIP encoders',
    'src/models/fusion.py': 'Fusion modules',
    'src/models/decoder.py': 'Point cloud decoder',
    'src/losses/__init__.py': 'Losses package init',
    'src/losses/losses.py': 'Loss functions',
    'configs/colab.yaml': 'Default config (PC)',
}

all_ok = True
for rel_path, desc in required_files.items():
    path = os.path.join(WORK_DIR, rel_path)
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ THIẾU'
    print(f'  {status}: {rel_path} ({desc})')
    if not exists:
        all_ok = False

if all_ok:
    print(f'\n✓ Project structure OK! ({len(required_files)} files)')
else:
    print('\n✗ Thiếu files! Upload toàn bộ thư mục src/ và configs/ lên Drive.')
    print(f'  Drive path: {WORK_DIR}/')

## 3. Cài Dependencies

In [ ]:
# Cài các thư viện cần thiết
!pip install -q open-clip-torch trimesh gradio plotly tensorboard scipy tqdm pyyaml open3d psutil

# Verify imports
import torch, open_clip, trimesh, plotly, scipy, psutil
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
try:
    import open3d
    print(f'Open3D: {open3d.__version__} (mesh chất lượng cao)')
except:
    print('Open3D: không cài được (dùng trimesh fallback)')

## 4. Tải dữ liệu Cap3D

Dữ liệu sẽ được tải và giải nén trực tiếp lên Drive (~40GB).  
Nếu đã có data, cell này sẽ skip.

In [ ]:
%%time
import os, zipfile, shutil, io, json

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
os.makedirs(f'{DATA_DIR}/pointclouds', exist_ok=True)
os.makedirs(f'{DATA_DIR}/rendered_images', exist_ok=True)

# --- [1/3] Captions ---
csv_path = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
if os.path.exists(csv_path):
    print('[1/3] ✓ Captions đã có')
else:
    print('[1/3] Tải captions...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ABO.csv \
        --repo-type dataset --local-dir /tmp/cap3d_download
    shutil.copy2('/tmp/cap3d_download/Cap3D_automated_ABO.csv', csv_path)
    print('✓ Captions saved')

# --- [2/3] Point clouds ---
pc_dir = f'{DATA_DIR}/pointclouds'
n_pc = len([f for f in os.listdir(pc_dir) if f.endswith('.ply')])
if n_pc > 100:
    print(f'\n[2/3] ✓ Point clouds đã có ({n_pc} files)')
else:
    print('\n[2/3] Tải point clouds (~1.5GB)...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D PointCloud_zips_ABO/compressed_pcs_00.zip \
        --repo-type dataset --local-dir /tmp/cap3d_download
    pc_zip = '/tmp/cap3d_download/PointCloud_zips_ABO/compressed_pcs_00.zip'
    print('Giải nén lên Drive...')
    with zipfile.ZipFile(pc_zip, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.ply')]
        for i, m in enumerate(members, 1):
            fname = os.path.basename(m)
            with z.open(m) as src, open(f'{pc_dir}/{fname}', 'wb') as dst:
                dst.write(src.read())
            if i % 2000 == 0 or i == len(members):
                print(f'  [{i}/{len(members)}]')
    os.remove(pc_zip)
    print(f'✓ Point clouds: {len(os.listdir(pc_dir))} files')

# --- [3/3] Rendered images (SMART RESUME) ---
img_dir = f'{DATA_DIR}/rendered_images'
existing_folders = set([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])

# Lấy danh sách object IDs từ captions
import pandas as pd
df = pd.read_csv(csv_path)
print(f'CSV columns: {list(df.columns)}')
# Tìm cột chứa object ID (có thể là 'object_id', 'object_id ', 'id', v.v.)
id_col = None
for col in df.columns:
    if 'object' in col.lower() or 'id' in col.lower():
        id_col = col
        break
if id_col is None:
    id_col = df.columns[0]  # Dùng cột đầu tiên nếu không tìm thấy
print(f'Using column: {id_col}')
expected_ids = set(df[id_col].astype(str).tolist())

missing_ids = expected_ids - existing_folders
n_existing = len(existing_folders)
n_missing = len(missing_ids)
n_total = len(expected_ids)

if n_missing == 0:
    print(f'\n[3/3] ✓ Rendered images đã có ({n_existing}/{n_total} folders)')
else:
    print(f'\n[3/3] Rendered images thiếu ({n_missing}/{n_total} folders). Tiếp tục tải...')
    !pip install -q huggingface_hub
    
    # Download main zip nếu chưa có trong /tmp
    main_zip_path = '/tmp/cap3d_download/RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip'
    if not os.path.exists(main_zip_path):
        print('Đang tải main zip (~37GB)...')
        !hf download tiange/Cap3D RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip \
            --repo-type dataset --local-dir /tmp/cap3d_download
        print('✓ Main zip downloaded')
    
    print(f'Giải nén {n_missing} folders còn thiếu...')
    with zipfile.ZipFile(main_zip_path, 'r') as outer_z:
        inner_zips = [m for m in outer_z.namelist() if m.endswith('.zip')]
        
        # Tạo map từ tên inner zip đến object_id
        zip_to_id = {z: os.path.basename(z).replace('.zip', '') for z in inner_zips}
        
        # Chỉ extract những zip cần thiết
        needed_zips = [z for z in inner_zips if zip_to_id[z] in missing_ids]
        print(f'  Cần giải nén {len(needed_zips)} inner zips...')
        
        for i, inner_name in enumerate(needed_zips, 1):
            obj_id = zip_to_id[inner_name]
            obj_dir = f'{img_dir}/{obj_id}'
            os.makedirs(obj_dir, exist_ok=True)
            
            inner_data = outer_z.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_data)) as inner_z:
                for img_file in inner_z.namelist():
                    if img_file.endswith(('.png', '.jpg')):
                        fname = os.path.basename(img_file)
                        with inner_z.open(img_file) as src, open(f'{obj_dir}/{fname}', 'wb') as dst:
                            dst.write(src.read())
            
            if i % 100 == 0 or i == len(needed_zips):
                print(f'  [{i}/{len(needed_zips)}]')
    
    # Xóa main zip để tiết kiệm disk
    if os.path.exists(main_zip_path):
        os.remove(main_zip_path)
        print('✓ Main zip deleted to save disk')
    
    n_final = len([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])
    print(f'✓ Rendered images: {n_final}/{n_total} folders')

# --- Tạo splits (LUÔN TẠO LẠI KHI DATA ĐẦY ĐỦ) ---
print('\nTạo splits từ data đầy đủ...')
os.chdir(WORK_DIR)
# Xóa splits cũ để tạo mới
splits_dir = f'{DATA_DIR}/splits'
if os.path.exists(splits_dir):
    shutil.rmtree(splits_dir)
!python src/data/prepare_data.py --data-dir "{DATA_DIR}" --skip-extract

for split in ['train', 'val', 'test']:
    sp = f'{DATA_DIR}/splits/{split}.txt'
    if os.path.exists(sp):
        with open(sp) as f:
            n = len([l for l in f if l.strip()])
        print(f'  {split}: {n} objects')

!df -h /content | tail -1
print('\n✓ Data sẵn sàng!')

## 5. Cấu hình Training

Config tự động tối ưu theo GPU:
- **A100 80GB**: Transformer decoder (width=384, 6 layers), batch=64
- **V100/T4 16GB**: Transformer decoder (width=256, 4 layers), batch=32
- **GPU nhỏ**: Transformer decoder (width=192, 3 layers), batch=16
- **CPU**: MLP decoder (fallback, nhẹ hơn), batch=16, 500 objects

**Transformer decoder** sử dụng CLIP grid features (49 spatial tokens) + cross-attention, cho kết quả 3D rõ hình thù hơn nhiều so với MLP decoder cũ.

In [ ]:
import yaml, torch, os

DATA_DIR = f'{WORK_DIR}/Cap3D_data'

# === Hardware info (chỉ hiển thị, KHÔNG thay đổi config) ===
has_gpu = torch.cuda.is_available()
if has_gpu:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {device_name} ({vram_gb:.0f} GB VRAM)')
else:
    print('⚠ Không có GPU! Model vẫn chạy nhưng rất chậm.')

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'System RAM: {ram_gb:.0f} GB')

# === CONFIG CỐ ĐỊNH: A100 High-RAM ===
# KHÔNG thay đổi theo GPU → checkpoint luôn tương thích khi đổi GPU
gpu_tier = 'A100 High-RAM (fixed)'
cfg_data = dict(batch_size=64, num_points=2048, num_workers=4, max_objects=0)
cfg_model = dict(coarse_points=256, fine_points=2048,
                 decoder_type='transformer', decoder_width=384,
                 decoder_layers=6, decoder_heads=6, fine_layers=3)
cfg_train = dict(epochs=100, lr=0.001, warmup_epochs=5, patience=25)

# Nếu GPU < 40GB VRAM, giảm batch_size để tránh OOM (model architecture giữ nguyên)
if has_gpu and vram_gb < 38:
    cfg_data['batch_size'] = 16
    cfg_data['num_workers'] = 2
    print(f'⚠ VRAM thấp ({vram_gb:.0f}GB) → batch_size giảm xuống 16 (model architecture giữ nguyên)')
elif has_gpu and vram_gb < 50:
    cfg_data['batch_size'] = 32
    print(f'ℹ VRAM trung bình ({vram_gb:.0f}GB) → batch_size = 32')

colab_config = {
    'data': {
        'data_dir': DATA_DIR,
        'num_points': cfg_data['num_points'],
        'image_size': 224,
        'batch_size': cfg_data['batch_size'],
        'num_workers': cfg_data['num_workers'],
        'max_objects': cfg_data['max_objects'],
    },
    'model': {
        'clip_model': 'ViT-B-32',
        'pretrained': 'laion2b_s34b_b79k',
        'freeze_clip': True,
        'decoder_type': cfg_model['decoder_type'],
        'decoder_width': cfg_model['decoder_width'],
        'decoder_layers': cfg_model['decoder_layers'],
        'decoder_heads': cfg_model['decoder_heads'],
        'fine_layers': cfg_model['fine_layers'],
        'fusion_type': 'concat',
        'fused_dim': 512,
        'coarse_points': cfg_model['coarse_points'],
        'fine_points': cfg_model['fine_points'],
    },
    'training': {
        'epochs': cfg_train['epochs'],
        'lr': cfg_train['lr'],
        'weight_decay': 0.01,
        'warmup_epochs': cfg_train['warmup_epochs'],
        'patience': cfg_train['patience'],
        'cooldown_seconds': 0,
    },
    'loss': {
        'lambda_cd': 1.0,
        'lambda_emd': 0.5,
        'lambda_color': 0.5,
        'lambda_repulsion': 0.1,
        'lambda_uniform': 0.05,
        'repulsion_k': 8,
        'repulsion_h': 0.03,
        'use_emd': False,
    },
    'logging': {
        'log_dir': 'runs',
        'checkpoint_dir': 'checkpoints',
        'log_every': 10,
        'val_every': 1,
    }
}

config_path = f'{WORK_DIR}/configs/colab.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

n_obj = cfg_data['max_objects'] if cfg_data['max_objects'] > 0 else 6343
batches = n_obj // cfg_data['batch_size']
print(f'\n✓ Config saved ({gpu_tier}):')
print(f'  decoder=transformer, width=384, layers=6+3, heads=6')
print(f'  batch={cfg_data["batch_size"]}, points=2048, epochs=100')
print(f'  objects={n_obj}, ~{batches} batches/epoch')
print(f'  loss: CD + Color + Repulsion + Uniformity')
print(f'  ⚡ Model architecture CỐ ĐỊNH → checkpoint tương thích mọi GPU')

## 6. Training

**Chống Colab timeout:** Chạy cell Keep-Alive TRƯỚC khi train.  
Checkpoints tự động lưu mỗi 50 batches trên Drive.

In [ ]:
# === CHỐNG COLAB TIMEOUT ===
# Chạy cell này TRƯỚC khi train. Nó sẽ chạy ngầm và giữ session sống.
# Cách hoạt động: inject JavaScript vào browser để click tự động mỗi 60 giây

from IPython.display import display, Javascript
display(Javascript('''
function KeepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    // Click vào nút connect nếu bị disconnect
    var buttons = document.querySelectorAll("colab-connect-button");
    if (buttons.length > 0) { buttons[0].click(); }
    // Click vào toolbar để giữ session active
    var toolbar = document.querySelector("#toolbar");
    if (toolbar) {
        var event = new MouseEvent('click', { bubbles: true });
        toolbar.dispatchEvent(event);
    }
}
// Chạy mỗi 60 giây
setInterval(KeepAlive, 60000);
console.log("✓ Keep-alive started (60s interval)");
'''))
print('✓ Keep-alive đã bật! Session sẽ không bị timeout.')

In [ ]:
%%time
# === BẮT ĐẦU TRAINING ===
import os, torch
os.chdir(WORK_DIR)

# Kiểm tra checkpoint để resume (xử lý file hỏng + model không khớp)
CKPT_DIR = f'{WORK_DIR}/checkpoints'
resume_path = ''

for ckpt_name in ['last_model.pth', 'best_model.pth']:
    ckpt_file = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(ckpt_file):
        try:
            ckpt = torch.load(ckpt_file, map_location='cpu')
            ckpt_cfg = ckpt.get('config', {})
            ckpt_points = ckpt_cfg.get('model', {}).get('fine_points', 0)
            ckpt_decoder = ckpt_cfg.get('model', {}).get('decoder_type', 'mlp')

            import yaml
            with open(f'{WORK_DIR}/configs/colab.yaml') as f:
                cur_cfg = yaml.safe_load(f)
            cur_points = cur_cfg.get('model', {}).get('fine_points', 0)
            cur_decoder = cur_cfg.get('model', {}).get('decoder_type', 'mlp')

            if ckpt_points == cur_points and ckpt_decoder == cur_decoder:
                resume_path = ckpt_file
                print(f'✓ Checkpoint tương thích: {ckpt_name} (decoder={ckpt_decoder}, points={ckpt_points})')
                break
            else:
                print(f'✗ Checkpoint không tương thích ({ckpt_name}):')
                print(f'  decoder: {ckpt_decoder} vs {cur_decoder}, fine_points: {ckpt_points} vs {cur_points}')
                print(f'  Xóa checkpoint cũ...')
                os.remove(ckpt_file)
        except Exception as e:
            print(f'✗ Checkpoint hỏng: {ckpt_name} → Xóa ({e})')
            os.remove(ckpt_file)

if resume_path:
    print(f'Resume từ: {resume_path}')
else:
    print('Train từ đầu (không có checkpoint tương thích)')

resume_flag = f'--resume "{resume_path}"' if resume_path else ''

!python src/train.py \
    --config configs/colab.yaml \
    {resume_flag}

In [ ]:
# Kiểm tra checkpoints đã lưu
import os
CKPT_DIR = f'{WORK_DIR}/checkpoints'
for f in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{f}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        print(f'✓ {f} ({size:.1f} MB)')
print('\nTraining hoàn tất! Checkpoints đã trên Drive.')

## 7. Evaluation

Đánh giá model trên test set bằng Chamfer Distance và F-Score.

In [ ]:
os.chdir(WORK_DIR)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

!python src/evaluate.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --split test \
    --output results/evaluation.json

In [ ]:
# Xem kết quả
import json
with open(f'{WORK_DIR}/results/evaluation.json') as f:
    results = json.load(f)
print('=== KẾT QUẢ EVALUATION ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

## 8. Demo Visualization

Test model trên test set với:
- Point cloud generation + post-processing (densify + smooth)
- Mesh reconstruction (Open3D Poisson / Ball Pivoting / Delaunay fallback)
- Interactive 3D visualization (Plotly)

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

import torch
import numpy as np
import yaml
from PIL import Image
from torchvision import transforms
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.models.multimodal3d import build_model
from src.demo import densify_point_cloud, smooth_point_cloud, point_cloud_to_mesh

# Load config & model
with open(f'{WORK_DIR}/configs/colab.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(config).to(device)

ckpt_path = f'{WORK_DIR}/checkpoints/best_model.pth'
if not os.path.exists(ckpt_path):
    ckpt_path = f'{WORK_DIR}/checkpoints/last_model.pth'

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
epoch = ckpt.get('epoch', '?')
best_cd = ckpt.get('best_val_cd', '?')
print(f'✓ Model loaded from {os.path.basename(ckpt_path)} (epoch={epoch}, best_cd={best_cd})')
print(f'Device: {device}')

In [ ]:
# === TEST VỚI NHIỀU OBJECTS TỪ TEST SET ===
import time

# Tiền xử lý
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

# Lấy test objects
test_split = f'{WORK_DIR}/Cap3D_data/splits/test.txt'
with open(test_split) as f:
    test_ids = [l.strip() for l in f if l.strip()]

# Load captions
import json
captions_path = f'{WORK_DIR}/Cap3D_data/captions.json'
with open(captions_path) as f:
    all_captions = json.load(f)

NUM_DEMO = min(4, len(test_ids))  # Hiển thị 4 objects

for idx in range(NUM_DEMO):
    obj_id = test_ids[idx]
    img_dir_obj = f'{WORK_DIR}/Cap3D_data/rendered_images/{obj_id}'
    img_files = sorted([f for f in os.listdir(img_dir_obj)
                        if f.endswith('.png') and '_depth' not in f and '_MatAlpha' not in f])

    if not img_files:
        print(f'[{idx+1}] {obj_id}: Không có ảnh, bỏ qua')
        continue

    img_path = os.path.join(img_dir_obj, img_files[0])
    img = Image.open(img_path).convert('RGB')

    # Caption
    caps = all_captions.get(obj_id, {})
    caption = caps.get('blip2', caps.get('caption', 'a 3D object'))
    if isinstance(caption, list):
        caption = caption[0]

    # Inference
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    text_tokens = model.tokenize([caption]).to(device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model(img_tensor, text_tokens)
    t_infer = time.time() - t0

    raw_pc = outputs['point_cloud'][0].cpu().numpy()

    # Post-processing: densify + smooth
    pc = densify_point_cloud(raw_pc, iterations=1, num_neighbors=3)
    pc = smooth_point_cloud(pc, num_neighbors=8, iterations=1)
    t_post = time.time() - t0 - t_infer

    # Mesh reconstruction
    mesh = point_cloud_to_mesh(pc)
    t_mesh = time.time() - t0 - t_infer - t_post

    xyz = pc[:, :3]
    rgb = np.clip(pc[:, 3:], 0, 1)
    colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb]

    n_faces = len(mesh.faces) if mesh and hasattr(mesh, 'faces') else 0

    print(f'\n[{idx+1}/{NUM_DEMO}] {obj_id}')
    print(f'  Caption: {caption[:80]}')
    print(f'  Points: {len(raw_pc)} → {len(pc)} | Mesh: {n_faces} faces')
    print(f'  Time: infer={t_infer:.1f}s, post={t_post:.1f}s, mesh={t_mesh:.1f}s')

    # Hiển thị ảnh input
    display(img.resize((224, 224)))

    # Point cloud visualization
    fig = go.Figure(data=[go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='markers',
        marker=dict(size=2.5, color=colors, opacity=0.95, line=dict(width=0)),
        hoverinfo='skip',
    )])
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False, showbackground=False),
            yaxis=dict(visible=False, showbackground=False),
            zaxis=dict(visible=False, showbackground=False),
            aspectmode='data',
            bgcolor='rgb(20,20,25)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
        ),
        paper_bgcolor='rgb(20,20,25)',
        margin=dict(l=0, r=0, b=0, t=30),
        height=450,
        title=dict(
            text=f'Point Cloud: {obj_id} ({len(pc)} pts)',
            font=dict(color='white', size=13),
        ),
    )
    fig.show()

    # Mesh visualization (nếu có)
    if mesh and hasattr(mesh, 'faces') and len(mesh.faces) > 10:
        verts = np.asarray(mesh.vertices)
        faces = np.asarray(mesh.faces)
        vc = None
        if hasattr(mesh.visual, 'vertex_colors') and mesh.visual.vertex_colors is not None:
            vc = np.asarray(mesh.visual.vertex_colors)[:, :3] / 255.0
            facecolor = [
                f'rgb({int(vc[f].mean(0)[0]*255)},{int(vc[f].mean(0)[1]*255)},{int(vc[f].mean(0)[2]*255)})'
                for f in faces
            ]
        else:
            facecolor = None

        fig2 = go.Figure(data=[go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            facecolor=facecolor,
            opacity=0.95,
            lighting=dict(ambient=0.5, diffuse=0.8, specular=0.3, roughness=0.5),
            lightposition=dict(x=100, y=200, z=300),
        )])
        fig2.update_layout(
            scene=dict(
                xaxis=dict(visible=False, showbackground=False),
                yaxis=dict(visible=False, showbackground=False),
                zaxis=dict(visible=False, showbackground=False),
                aspectmode='data',
                bgcolor='rgb(20,20,25)',
                camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            ),
            paper_bgcolor='rgb(20,20,25)',
            margin=dict(l=0, r=0, b=0, t=30),
            height=450,
            title=dict(
                text=f'Mesh: {obj_id} ({len(mesh.vertices)} verts, {n_faces} faces)',
                font=dict(color='white', size=13),
            ),
        )
        fig2.show()

print(f'\n✓ Demo hoàn tất! ({NUM_DEMO} objects)')

## 9. Gradio Demo (Interactive UI)

Chạy Gradio demo với UI chuyên nghiệp:
- Upload ảnh + nhập text → tạo 3D
- 3 mức chất lượng: low / medium / high
- Xuất .ply, .obj, .glb
- Public link để chia sẻ

In [ ]:
# Chạy Gradio demo (tạo public link để truy cập từ browser)
os.chdir(WORK_DIR)

!python src/demo.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --share

## 10. Download Checkpoint về máy

Tải checkpoint đã train về máy để chạy demo local.

In [ ]:
# Download checkpoint về máy
from google.colab import files
import os

CKPT_DIR = f'{WORK_DIR}/checkpoints'

for ckpt_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        print(f'✓ {ckpt_name} ({size:.1f} MB)')

# Chọn checkpoint để download
ckpt_to_download = f'{CKPT_DIR}/best_model.pth'
if os.path.exists(ckpt_to_download):
    print(f'\nDownloading {ckpt_to_download}...')
    files.download(ckpt_to_download)
else:
    ckpt_to_download = f'{CKPT_DIR}/last_model.pth'
    if os.path.exists(ckpt_to_download):
        print(f'\nDownloading {ckpt_to_download}...')
        files.download(ckpt_to_download)
    else:
        print('✗ Không tìm thấy checkpoint nào!')

## 11. Ghi chú & Hướng dẫn

### Cấu trúc dự án trên Drive:
```
ThiGiacMayTinh_BaiTapLon/
├── src/
│   ├── train.py          # Training script
│   ├── evaluate.py       # Evaluation (CD, F-Score)
│   ├── demo.py           # Gradio demo + mesh reconstruction
│   ├── data/
│   │   ├── dataset.py    # Dataset & DataLoader
│   │   ├── prepare_data.py
│   │   └── transforms.py
│   ├── models/
│   │   ├── multimodal3d.py         # Main model (CLIP + Decoder)
│   │   ├── encoders.py             # CLIP encoders (CLS + Grid features)
│   │   ├── fusion.py               # Concat/CrossAttention fusion (MLP decoder)
│   │   ├── decoder.py              # MLP decoder (legacy)
│   │   └── transformer_decoder.py  # Transformer decoder (mới, chất lượng cao)
│   └── losses/
│       └── losses.py     # Chamfer Distance, EMD, Color loss
├── configs/
│   ├── default.yaml      # Config cho PC local (MLP decoder)
│   └── colab.yaml        # Config cho Colab (Transformer decoder)
├── Cap3D_data/           # Dataset (~40GB)
├── checkpoints/          # Model checkpoints
└── results/              # Evaluation results
```

### Transformer vs MLP decoder:
| | **MLP decoder** | **Transformer decoder** |
|---|---|---|
| Conditioning | 1 vector 512-dim | **49 grid tokens** (spatial) |
| Points communication | Không | **Self-attention** (coarse stage) |
| Detail | Từ ảnh | **Cross-attention** đến từng vùng ảnh |
| Trainable params | ~5M | **~15-20M** |
| Chất lượng output | Thấp (bẹt, mờ) | **Cao (rõ hình thù)** |

### Nếu Colab bị ngắt giữa chừng:
1. Reconnect Colab
2. Chạy lại: Cell 1 (Mount) → Cell 3 (Deps) → Cell 5 (Config) → Cell 6 (Train)
3. Training sẽ tự **resume** từ checkpoint trên Drive
4. Checkpoint save mỗi 50 batches → mất tối đa ~vài phút progress

### Auto-detect GPU config:
| GPU | VRAM | decoder | width | layers | batch | ~Tốc độ |
|---|---|---|---|---|---|---|
| **A100** | 40-80GB | transformer | 384 | 6+3 | 64 | ~20-30 phút/epoch |
| **V100/T4** | 16GB | transformer | 256 | 4+2 | 32 | ~90-120 phút/epoch |
| **GPU nhỏ** | <14GB | transformer | 192 | 3+1 | 16 | ~60-90 phút/epoch |
| **CPU** | - | mlp | - | - | 16 | ~5-10 phút/epoch |

### Chạy demo trên máy local:
```bash
python src/demo.py --config configs/colab.yaml --checkpoint checkpoints/best_model.pth
```